# Стелла 1.5B — обучение в бесплатном Colab

**Сначала:** Среда выполнения → Сменить среду → **T4 GPU**. Потом Среда выполнения → Выполнить всё.

Обучение ~15–20 минут. В конце получите файл `stella-1.5b-Q8_0.gguf` (~1,6 ГБ) — его запускаете у себя в Ollama, видеокарта для этого не нужна.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Код и зависимости

In [ ]:
!git clone -q -b claude/tender-cray-foqpnc https://github.com/a1x10/project3
%cd project3/llm_model
!pip install -q peft sentencepiece

## 2. Датасет

Свои обычные диалоги (JSONL, поле `messages`) можно загрузить через панель «Файлы» в `project3/llm_model/` и указать в `EXTRA`.

In [ ]:
EXTRA = ''  # например 'my_dialogs.jsonl'
!python build_dataset.py --identity 800 {'--extra ' + EXTRA if EXTRA else ''}

## 3. Обучение (LoRA)

In [ ]:
!python train_sft.py --batch 8 --grad-accum 2 --epochs 3

## 4. Слияние и проверка

In [ ]:
!python export.py --adapter runs/stella-lora --out stella-1.5b
!python chat.py --model stella-1.5b
!python chat.py --model stella-1.5b --no-system 2>/dev/null | head -8

## 5. GGUF для Ollama

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!python /content/llama.cpp/convert_hf_to_gguf.py stella-1.5b --outtype q8_0 --outfile stella-1.5b-Q8_0.gguf
!ls -lh *.gguf

## 6. Забрать файл

Через Google Диск надёжнее, чем прямое скачивание из браузера (файл большой).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/stella
!cp stella-1.5b-Q8_0.gguf Modelfile /content/drive/MyDrive/stella/
print('Готово: Google Диск → stella/')